# Exercício 01 — PromptLab Consultoria

**Versão didática:** o notebook contém **schemas**, **prompts**, **`gerar_resposta`** e **cinco modos de coordenação multi-agente** sem `from app…`. A pasta [`app/`](app/) mantém o mesmo código para a API FastAPI.

Documentação: [`docs/`](docs/).

## 0. Ambiente e caminhos

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

EX_ROOT = Path.cwd().resolve()
REPO_ROOT = EX_ROOT.parent.parent

load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(EX_ROOT / ".env", override=True)

if not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Defina GOOGLE_API_KEY no .env na raiz do repositório.")

print("OK — Repo:", REPO_ROOT)
print("OK — Exercício:", EX_ROOT)

## 1. Pydantic — entrada e saída (`PerfilAssistente`)

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field

class PerfilAssistente(str, Enum):
    tecnico = "tecnico"
    professor = "professor"
    comercial = "comercial"
    sarcastico_nerd = "sarcastico_nerd"

class PerguntaEntrada(BaseModel):
    perfil: PerfilAssistente
    pergunta: str = Field(..., min_length=1, max_length=8000)

class RespostaSaida(BaseModel):
    perfil: str
    resposta: str

ex = PerguntaEntrada(perfil="professor", pergunta="O que é um agente de IA?")
print(ex.model_dump_json(indent=2))

## 2. Prompts por perfil + `gerar_resposta` (Gemini)

Cada perfil tem uma **mensagem de sistema** própria; aplicamos **retry** simples em quotas (429).

In [ ]:
import os
import time

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

_DEFAULT_MODEL = "gemini-2.0-flash"

def nome_modelo() -> str:
    raw = (os.environ.get("GEMINI_MODEL") or _DEFAULT_MODEL).strip() or _DEFAULT_MODEL
    return raw.removeprefix("models/")

def criar_chat(temperature: float = 0.4) -> ChatGoogleGenerativeAI:
    return ChatGoogleGenerativeAI(model=nome_modelo(), temperature=temperature)

SYSTEM_PROMPTS: dict[PerfilAssistente, str] = {
    PerfilAssistente.tecnico: (
        "És um consultor técnico sénior da PromptLab Consultoria. "
        "Respondes em português europeu, com precisão, estrutura clara (tópicos quando ajudar) "
        "e vocabulário técnico adequado. Evitas jargão desnecessário; quando usares um termo obscureto, "
        "define-o numa frase. Não inventes factos: se faltar contexto, pede esclarecimento brevemente."
    ),
    PerfilAssistente.professor: (
        "És um professor didático da PromptLab Consultoria. "
        "Respondes em português europeu com analogias simples, passos curtos e um mini-resumo final. "
        "Adaptas o nível ao que o utilizador parece precisar; incentivas o raciocínio sem ser condescendente."
    ),
    PerfilAssistente.comercial: (
        "És um atendente comercial da PromptLab Consultoria. "
        "Respondes em português europeu com tom cordial, positivo e orientado a valor para o cliente. "
        "Destacas benefícios práticos quando fizer sentido; manténs respostas concisas e profissionais."
    ),
    PerfilAssistente.sarcastico_nerd: (
        "És o consultor sarcástico nerd da PromptLab Consultoria (desafio extra). "
        "Respondes em português europeu com humor nerd leve e ironia corporativa educada — "
        "nunca cruel nem desrespeitoso. Continuas a ser útil: dás a resposta certa antes ou depois da piada."
    ),
}

def obter_system_prompt(perfil: PerfilAssistente) -> str:
    return SYSTEM_PROMPTS[perfil]

def _eh_quota(exc: BaseException) -> bool:
    t = str(exc).upper()
    return "429" in t or "RESOURCE_EXHAUSTED" in t

def gerar_resposta(perfil: PerfilAssistente, pergunta: str) -> str:
    llm = criar_chat()
    mensagens = [
        SystemMessage(content=obter_system_prompt(perfil)),
        HumanMessage(content=pergunta),
    ]
    max_t = max(1, int(os.environ.get("GEMINI_RETRY_ATTEMPTS", "5")))
    base = max(0.5, float(os.environ.get("GEMINI_RETRY_DELAY_SEC", "2")))
    ultimo: BaseException | None = None
    for tentativa in range(max_t):
        try:
            res: AIMessage = llm.invoke(mensagens)
            if isinstance(res.content, str):
                return res.content.strip()
            return str(res.content).strip()
        except Exception as e:
            ultimo = e
            if _eh_quota(e) and tentativa < max_t - 1:
                time.sleep(min(base * (2**tentativa), 60.0))
                continue
            raise
    assert ultimo is not None
    raise ultimo

for p in PerfilAssistente:
    body = PerguntaEntrada(perfil=p, pergunta="O que é RAG?")
    out = RespostaSaida(perfil=body.perfil.value, resposta=gerar_resposta(body.perfil, body.pergunta))
    print("---", out.perfil, "---")
    print(out.resposta[:900])

## 3. Coordenação multi-agente — cinco modos

| Modo | Ideia |
|------|------|
| `sequencial_pipeline` | técnico → professor → comercial |
| `paralelo_sintese` | 3 perfis em paralelo + sintetizador |
| `debate_critico` | técnico → sarcástico critica → técnico revisa |
| `router_inteligente` | LLM escolhe um perfil |
| `refinamento_triplo` | professor → técnico → comercial |

O bloco seguinte define **schemas de coordenação**, prompts do router/sintetizador e **`executar_coordenacao`**.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from enum import Enum

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field

SYSTEM_ROUTER = 'És um router de pedidos da PromptLab Consultoria. Analisas a pergunta do cliente e escolhes **exatamente um** perfil interno para responder sozinho: `tecnico` (profundidade técnica, arquitetura), `professor` (didático, passo a passo), `comercial` (valor de negócio, próximos passos), `sarcastico_nerd` (humor leve mas resposta útil). Devolves também uma justificação curta.'

SYSTEM_SINTETIZADOR = 'És o coordenador editorial da PromptLab Consultoria. Recebes respostas ao mesmo pedido vindas de vários consultores com estilos diferentes. Produzes uma **única resposta final** em português europeu que: (1) integra os pontos corretos sem contradições; (2) mantém tom profissional neutro; (3) indica brevemente quando houve tensão entre perspetivas. Não cries nomes de produtos ou dados inventados.'

class ModoCoordenacao(str, Enum):
    """Padrões de coordenação entre agentes (cada um usa um perfil de sistema diferente)."""

    sequencial_pipeline = "sequencial_pipeline"
    paralelo_sintese = "paralelo_sintese"
    debate_critico = "debate_critico"
    router_inteligente = "router_inteligente"
    refinamento_triplo = "refinamento_triplo"


class CoordenacaoEntrada(BaseModel):
    pergunta: str = Field(..., min_length=1, max_length=8000)
    modo: ModoCoordenacao = Field(
        ...,
        description="Estratégia de coordenação entre agentes (perfis).",
    )


class EtapaCoordenacao(BaseModel):
    """Um passo observável na orquestração."""

    agente: str = Field(..., description="Identificador do agente / perfil.")
    papel: str = Field(..., description="Função nesta coordenação.")
    saida: str = Field(..., description="Texto produzido nesta etapa.")


class CoordenacaoSaida(BaseModel):
    modo: str
    resposta_final: str
    etapas: list[EtapaCoordenacao] = Field(default_factory=list)

def _invoke_custom(system: str, human: str, temperature: float = 0.35) -> str:
    llm = criar_chat(temperature=temperature)
    res: AIMessage = llm.invoke(
        [SystemMessage(content=system), HumanMessage(content=human)]
    )
    if isinstance(res.content, str):
        return res.content.strip()
    return str(res.content).strip()


class _RouterSchema(BaseModel):
    perfil_escolhido: str = Field(
        description="Um de: tecnico, professor, comercial, sarcastico_nerd"
    )
    motivo_breve: str = Field(description="Uma frase.")


def _sequencial_pipeline(pergunta: str) -> CoordenacaoSaida:
    etapas: list[EtapaCoordenacao] = []

    t1 = gerar_resposta(PerfilAssistente.tecnico, pergunta)
    etapas.append(
        EtapaCoordenacao(agente="tecnico", papel="Rascunho técnico inicial", saida=t1)
    )

    t2 = gerar_resposta(
        PerfilAssistente.professor,
        "Texto técnico do colega:\n"
        f"{t1}\n\n"
        "Reformula para um público intermédio: mantém rigor mas melhora pedagogia.",
    )
    etapas.append(
        EtapaCoordenacao(
            agente="professor",
            papel="Reformulação didática do output técnico",
            saida=t2,
        )
    )

    t3 = gerar_resposta(
        PerfilAssistente.comercial,
        "Versão didática:\n"
        f"{t2}\n\n"
        "Acrescenta valor para decisão de negócio (benefício, próximo passo sugerido) sem alterar factos.",
    )
    etapas.append(
        EtapaCoordenacao(
            agente="comercial",
            papel="Camada de valor e próximos passos",
            saida=t3,
        )
    )

    return CoordenacaoSaida(
        modo=ModoCoordenacao.sequencial_pipeline.value,
        resposta_final=t3,
        etapas=etapas,
    )


def _paralelo_sintese(pergunta: str) -> CoordenacaoSaida:
    etapas: list[EtapaCoordenacao] = []

    def _run(perfil: PerfilAssistente, papel: str) -> tuple[str, str, str]:
        texto = gerar_resposta(perfil, pergunta)
        return perfil.value, papel, texto

    specs = [
        (PerfilAssistente.tecnico, "Perspetiva técnica"),
        (PerfilAssistente.professor, "Perspetiva pedagógica"),
        (PerfilAssistente.comercial, "Perspetiva comercial"),
    ]
    blocos: dict[str, str] = {}

    with ThreadPoolExecutor(max_workers=3) as pool:
        fut_map = {
            pool.submit(_run, perfil, papel): (perfil, papel)
            for perfil, papel in specs
        }
        for fut in as_completed(fut_map):
            pid, papel, texto = fut.result()
            etapas.append(EtapaCoordenacao(agente=pid, papel=papel, saida=texto))
            blocos[pid] = texto

    _ordem = {"tecnico": 0, "professor": 1, "comercial": 2}
    etapas.sort(key=lambda e: _ordem.get(e.agente, 99))

    pacote = (
        f"Pergunta original:\n{pergunta}\n\n"
        f"### Técnico\n{blocos['tecnico']}\n\n"
        f"### Professor\n{blocos['professor']}\n\n"
        f"### Comercial\n{blocos['comercial']}"
    )
    final = _invoke_custom(
        SYSTEM_SINTETIZADOR,
        "Funde as perspetivas numa resposta única ao cliente:\n\n" + pacote,
        temperature=0.3,
    )
    etapas.append(
        EtapaCoordenacao(
            agente="sintetizador",
            papel="Fusão das três perspetivas paralelas",
            saida=final,
        )
    )

    return CoordenacaoSaida(
        modo=ModoCoordenacao.paralelo_sintese.value,
        resposta_final=final,
        etapas=etapas,
    )


def _debate_critico(pergunta: str) -> CoordenacaoSaida:
    etapas: list[EtapaCoordenacao] = []

    r1 = gerar_resposta(PerfilAssistente.tecnico, pergunta)
    etapas.append(
        EtapaCoordenacao(agente="tecnico", papel="Primeira resposta técnica", saida=r1)
    )

    critica = gerar_resposta(
        PerfilAssistente.sarcastico_nerd,
        "Pergunta:\n"
        f"{pergunta}\n\n"
        "Resposta técnica do colega:\n"
        f"{r1}\n\n"
        "Identifica lacunas, ambiguidades ou otimismo excessivo. Mantém humor leve.",
    )
    etapas.append(
        EtapaCoordenacao(
            agente="sarcastico_nerd",
            papel="Crítica construtiva / segunda opinião",
            saida=critica,
        )
    )

    r2 = gerar_resposta(
        PerfilAssistente.tecnico,
        "Pergunta:\n"
        f"{pergunta}\n\n"
        "A minha primeira resposta:\n"
        f"{r1}\n\n"
        "Crítica recebida:\n"
        f"{critica}\n\n"
        "Resposta técnica **revisada** que integra o feedback.",
    )
    etapas.append(
        EtapaCoordenacao(
            agente="tecnico",
            papel="Revisão técnica pós-debate",
            saida=r2,
        )
    )

    return CoordenacaoSaida(
        modo=ModoCoordenacao.debate_critico.value,
        resposta_final=r2,
        etapas=etapas,
    )


def _router_inteligente(pergunta: str) -> CoordenacaoSaida:
    llm = criar_chat(temperature=0.1).with_structured_output(_RouterSchema)
    decisao: _RouterSchema = llm.invoke(
        [
            SystemMessage(content=SYSTEM_ROUTER),
            HumanMessage(content=pergunta),
        ]
    )
    etapas = [
        EtapaCoordenacao(
            agente="router",
            papel="Escolha do perfil único",
            saida=f"{decisao.perfil_escolhido}: {decisao.motivo_breve}",
        )
    ]
    raw = (
        decisao.perfil_escolhido.strip()
        .lower()
        .replace("á", "a")
        .replace("í", "i")
        .replace("-", "_")
        .replace(" ", "_")
    )
    aliases = {
        "sarcastico": "sarcastico_nerd",
        "nerd": "sarcastico_nerd",
        "comercial_vendas": "comercial",
    }
    raw = aliases.get(raw, raw)
    try:
        perfil = PerfilAssistente(raw)
    except ValueError:
        perfil = PerfilAssistente.tecnico
        etapas.append(
            EtapaCoordenacao(
                agente="router",
                papel="Fallback",
                saida="Perfil inválido devolvido pelo router — usado `tecnico`.",
            )
        )

    texto = gerar_resposta(perfil, pergunta)
    etapas.append(
        EtapaCoordenacao(
            agente=perfil.value,
            papel="Resposta final pelo perfil escolhido",
            saida=texto,
        )
    )

    return CoordenacaoSaida(
        modo=ModoCoordenacao.router_inteligente.value,
        resposta_final=texto,
        etapas=etapas,
    )


def _refinamento_triplo(pergunta: str) -> CoordenacaoSaida:
    """Professor abre (acessível) → técnico endurece rigor → comercial fecha."""
    etapas: list[EtapaCoordenacao] = []

    s1 = gerar_resposta(PerfilAssistente.professor, pergunta)
    etapas.append(
        EtapaCoordenacao(
            agente="professor",
            papel="Primeira explicação didática",
            saida=s1,
        )
    )

    s2 = gerar_resposta(
        PerfilAssistente.tecnico,
        "Versão didática inicial:\n"
        f"{s1}\n\n"
        "Verifica rigor técnico, corrige imprecisões e acrescenta nuances necessárias.",
    )
    etapas.append(
        EtapaCoordenacao(
            agente="tecnico",
            papel="Endurecimento técnico / correções",
            saida=s2,
        )
    )

    s3 = gerar_resposta(
        PerfilAssistente.comercial,
        "Versão técnica refinada:\n"
        f"{s2}\n\n"
        "Apresenta ao cliente final de forma clara, com ênfase em valor e próximo passo.",
    )
    etapas.append(
        EtapaCoordenacao(
            agente="comercial",
            papel="Empacotamento final para cliente",
            saida=s3,
        )
    )

    return CoordenacaoSaida(
        modo=ModoCoordenacao.refinamento_triplo.value,
        resposta_final=s3,
        etapas=etapas,
    )


def executar_coordenacao(modo: ModoCoordenacao, pergunta: str) -> CoordenacaoSaida:
    if modo == ModoCoordenacao.sequencial_pipeline:
        return _sequencial_pipeline(pergunta)
    if modo == ModoCoordenacao.paralelo_sintese:
        return _paralelo_sintese(pergunta)
    if modo == ModoCoordenacao.debate_critico:
        return _debate_critico(pergunta)
    if modo == ModoCoordenacao.router_inteligente:
        return _router_inteligente(pergunta)
    if modo == ModoCoordenacao.refinamento_triplo:
        return _refinamento_triplo(pergunta)
    raise ValueError(modo)


DESCRICOES_MODOS: dict[str, str] = {
    ModoCoordenacao.sequencial_pipeline.value: (
        "Pipeline sequencial: técnico → professor (didática) → comercial (valor)."
    ),
    ModoCoordenacao.paralelo_sintese.value: (
        "Paralelo: três perfis respondem em paralelo; um sintetizador funde num texto único."
    ),
    ModoCoordenacao.debate_critico.value: (
        "Debate: técnico responde, perfil sarcástico critica, técnico revisa."
    ),
    ModoCoordenacao.router_inteligente.value: (
        "Router LLM escolhe um único perfil; esse agente responde."
    ),
    ModoCoordenacao.refinamento_triplo.value: (
        "Refinamento: professor abre → técnico corrige rigor → comercial fecha."
    ),
}

### Demonstração rápida — `router_inteligente`

In [ ]:
pergunta_coord = "Quando usar RAG em vez de só aumentar o contexto do modelo?"

out_router = executar_coordenacao(ModoCoordenacao.router_inteligente, pergunta_coord)
print("Etapas:")
for e in out_router.etapas:
    print("-", e.agente, ":", e.papel)
print("\n--- RESPOSTA FINAL (trecho) ---\n", out_router.resposta_final[:900])

### Opcional — `debate_critico` (mais chamadas ao modelo)

In [ ]:
# out_debate = executar_coordenacao(ModoCoordenacao.debate_critico, pergunta_coord)
# print(out_debate.resposta_final[:1200])
print("Descomente as linhas acima para correr o modo debate_critico completo.")

## Referências

- [`app/chains/coordenacoes.py`](app/chains/coordenacoes.py) — espelho modular
- [`app/chains/promptlab_chain.py`](app/chains/promptlab_chain.py)
- [`docs/`](docs/)